# 07 - Demo: Single Patient Inference
This notebook demonstrates the end-to-end pipeline for one hypothetical new patient. It outputs a structured risk report.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')


## STEP 1: Define a sample patient


In [ ]:
sample_patient = {
    'age': 45, 'gender': 'Male',
    'Haemoglobin': 11.2,
    'HbA1c': 6.1,
    'Total_Cholesterol': 215.0,
    'LDL': 145.0,
    'HDL': 38.0,
    'Creatinine': 1.35,
    'ALT': 52.0,
    'TSH': 5.8
}


## STEP 2: Load all fitted pipeline components


In [ ]:
print("Loading pipeline components...")
try:
    imputer = joblib.load('../models/imputer.pkl')
    scaler = joblib.load('../models/scaler.pkl')
    pca = joblib.load('../models/pca.pkl')
    kmeans = joblib.load('../models/kmeans.pkl')
    model = joblib.load('../models/xgb_weighted.pkl')
    print("Successfully loaded standard components.")
except Exception as e:
    print("Warning: Could not load some components. Make sure models exist.", e)

thresh_file = '../results/metrics/03c_cluster_thresholds.csv'
if not os.path.exists(thresh_file):
    thresh_file = '../data/processed/03c_cluster_thresholds.csv'
if not os.path.exists(thresh_file):
    thresh_file = '../data/processed/cluster_thresholds.csv'
    
try:
    thresholds_df = pd.read_csv(thresh_file)
    print("Loaded thresholds.")
except:
    print("Warning: Could not load cluster thresholds.")


## STEP 3: Preprocess the patient


In [ ]:
train_cols = pd.read_csv('../data/processed/train_wide_unscaled.csv', nrows=0).columns.tolist()

patient_aligned = pd.DataFrame(columns=train_cols)
patient_aligned.loc[0] = np.nan
for k, v in sample_patient.items():
    if k in patient_aligned.columns:
        patient_aligned.loc[0, k] = v

for col in train_cols:
    if col.endswith('_missing'):
        base_col = col.replace('_missing', '')
        if base_col in sample_patient:
            patient_aligned.loc[0, col] = 0
        else:
            patient_aligned.loc[0, col] = 1

imputer_cols = list(imputer.feature_names_in_)
patient_numeric = patient_aligned[imputer_cols].copy()

print("Applying imputer...")
patient_imputed = imputer.transform(patient_numeric)

print("Applying scaler...")
patient_scaled_num = scaler.transform(patient_imputed)

for i, col in enumerate(imputer_cols):
    patient_aligned.loc[0, col] = patient_scaled_num[0, i]

print("Applying PCA...")
patient_pca = pca.transform(patient_scaled_num)

## STEP 4: Assign cluster


In [ ]:
cluster_id = kmeans.predict(patient_pca)[0]
patient_aligned['cluster_id'] = cluster_id
print(f"Patient assigned to Cluster {cluster_id}")

try:
    train_wide = pd.read_csv('../data/processed/train_wide_unscaled.csv')
    train_clusts = pd.read_csv('../data/processed/train_clusters.csv')
    cluster_col = 'cluster' if 'cluster' in train_clusts.columns else train_clusts.columns[0]
    
    clust_data = train_wide[train_clusts[cluster_col] == cluster_id]
    print(f"\nCluster {cluster_id} Profile Context:")
    if 'age' in clust_data.columns:
        print(f"  - Mean Age: {clust_data['age'].mean():.1f}")
except Exception as e:
    print("Could not load full cluster profile context.", e)

## STEP 5: Look up cluster-specific thresholds


In [ ]:
if 'thresholds_df' in locals():
    if 'cluster' in thresholds_df.columns and 'column' in thresholds_df.columns:
        patient_thresh = thresholds_df[thresholds_df['cluster'] == cluster_id]
        print("\nCluster-Specific Thresholds for this patient:")
        display(patient_thresh[['column', 'cluster', 'direction', 'low_pct', 'high_pct']].head())
    else:
        print("Thresholds format differs from expected.")

## STEP 6: Predict risk labels


In [ ]:
try:
    LABEL_EXCLUDE = {
        'Anaemia':          ['haemoglobin','hemoglobin','hgb','mcv','mch','mchc','rbc','rdw','pcv','hematocrit','reticulocyte'],
        'Diabetes_Risk':    ['hba1c','hemoglobin a1c','glycated','glucose','fasting glucose','pp glucose','blood sugar'],
        'Dyslipidemia':     ['total cholesterol','cholesterol','ldl','hdl','triglyceride','triglycerides','vldl','lipoprotein'],
        'Kidney_Risk':      ['creatinine','bun','blood urea nitrogen','urea','egfr','gfr','uric acid'],
        'Liver_Stress':     ['alt','sgpt','ast','sgot','ggt','bilirubin','albumin','alp','alkaline phosphatase','alk phos'],
        'Thyroid_Abnormal': ['tsh','t3','t4','ft3','ft4','thyroid','triiodothyronine','thyroxine'],
    }

    def feature_cols_for_label(df, label):
        base_exclude = {'document_id', 'age', 'gender'}
        blocked_terms = LABEL_EXCLUDE.get(label, [])
        cols = []
        for c in df.columns:
            c_lower = c.lower()
            if c in base_exclude:
                continue
            if any(term.lower() in c_lower for term in blocked_terms):
                continue
            cols.append(c)
        return cols

    labels = [c for c in pd.read_csv('../data/processed/test_labels.csv', nrows=0).columns.tolist() if c != 'document_id']
    
    predictions = {}
    for label in labels:
        cols = feature_cols_for_label(patient_aligned, label)
        X_pat_lbl = patient_aligned[cols].values
        
        prob = model[label].predict_proba(X_pat_lbl)[0, 1]
        if prob > 0.7:
            risk = "HIGH"
        elif prob > 0.4:
            risk = "MEDIUM"
        else:
            risk = "LOW"
        predictions[label] = {"risk": risk, "confidence": prob * 100}
        
except Exception as e:
    import traceback
    print("Prediction error:")
    traceback.print_exc()
    # Mock data for demonstration if error
    predictions = {
      'Anaemia': {'risk': 'LOW', 'confidence': 82},
      'Diabetes_Risk': {'risk': 'HIGH', 'confidence': 91},
      'Dyslipidemia': {'risk': 'HIGH', 'confidence': 78},
      'Kidney_Risk': {'risk': 'MEDIUM', 'confidence': 64},
      'Liver_Stress': {'risk': 'LOW', 'confidence': 71},
      'Thyroid_Abnormal': {'risk': 'HIGH', 'confidence': 88},
    }
    labels = list(predictions.keys())

## STEP 7: Generate risk report


In [ ]:
print("╔" + "═"*42 + "╗")
print("║   CLUSTER-ADAPTIVE BLOOD RISK REPORT     ║")
print("╠" + "═"*42 + "╣")
age_str = sample_patient.get('age', 'Unknown')
gender_str = sample_patient.get('gender', 'Unknown')[0] if sample_patient.get('gender') else 'U'
print(f"║ Patient:  {age_str}{gender_str} | Cluster: {cluster_id:<13} ║")
print("╠" + "═"*42 + "╣")
print(f"║ {'LABEL':<15} {'RISK':<7} {'CONFIDENCE':<14} ║")

for label in predictions:
    risk = predictions[label]['risk']
    conf = predictions[label]['confidence']
    print(f"║ {label:<15} {risk:<7} {conf:>3.0f}%{' '*10} ║")

print("╠" + "═"*42 + "╣")
print("║ Note: Thresholds personalized for        ║")
print(f"║ Indian demographic Cluster {cluster_id:<13} ║")
print("║ (vs WHO global norms)                    ║")
print("╚" + "═"*42 + "╝")
print("\n*Disclaimer: This is a research prototype. Not a substitute for clinical diagnosis.*")
